<a href="https://colab.research.google.com/github/Srujana767/Snap2Meal/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install dependencies
!pip install -q openai gradio pillow


In [2]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import base64
import io
import tempfile

import gradio as gr
from openai import OpenAI
from PIL import Image

from google.colab import userdata

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [3]:
# ============================================================
# CELL 2: CONNECT TO OPENROUTER
# ============================================================

try:
    # Get API key from Google Colab Secrets
    api_key = userdata.get("OPENROUTER_API_KEY")

    # Create OpenRouter client
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
        timeout=45.0
    )

    print("✅ OpenRouter connected successfully")

except Exception as e:
    raise ValueError(
        "❌ Make sure OPENROUTER_API_KEY is added "
        "in Colab Secrets and the toggle is ON!"
    )

✅ OpenRouter connected successfully


In [4]:
# ============================================================
# CELL 3: CUSTOM CSS
# ============================================================

custom_css = """
.markdown-text table {
    width: 100%;
    border-collapse: collapse;
    margin: 15px 0;
}

.markdown-text th {
    background-color: #4f46e5;
    color: white;
    padding: 10px;
    text-align: left;
}

.markdown-text td {
    border: 1px solid #e5e7eb;
    padding: 8px 12px;
}
"""

print("✅ CSS loaded")

✅ CSS loaded


In [5]:
# ============================================================
# CELL 4: IMAGE PROCESSING + AI RECIPE GENERATION
# ============================================================

def analyze_pantry(
    image,
    extra_ingredients,
    cuisine_pref,
    dietary_pref,
    meal_types,
    servings
):

    # --------------------------------------------------------
    # STEP 1: CHECK IMAGE
    # --------------------------------------------------------

    if image is None:
        return "❌ Please upload an image first!", None

    try:

        # ----------------------------------------------------
        # STEP 2: CONVERT IMAGE TO PIL
        # ----------------------------------------------------

        pil_img = Image.fromarray(image)

        # Resize image to reduce API request size
        pil_img.thumbnail((512, 512))


        # ----------------------------------------------------
        # STEP 3: CONVERT IMAGE TO JPEG
        # ----------------------------------------------------

        buffered = io.BytesIO()

        pil_img.save(
            buffered,
            format="JPEG",
            quality=80
        )


        # ----------------------------------------------------
        # STEP 4: CONVERT IMAGE TO BASE64
        # ----------------------------------------------------

        base64_image = base64.b64encode(
            buffered.getvalue()
        ).decode("utf-8")


        # ----------------------------------------------------
        # STEP 5: PREPARE USER PREFERENCES
        # ----------------------------------------------------

        pref_text = (
            dietary_pref
            if dietary_pref
            else "None"
        )

        types_text = (
            ", ".join(meal_types)
            if meal_types
            else "Any Meal Type"
        )

        extra_text = (
            extra_ingredients
            if extra_ingredients.strip()
            else "None extra provided."
        )

        cuisine_text = (
            cuisine_pref
            if cuisine_pref
            else "Any Cuisine"
        )


        # ----------------------------------------------------
        # STEP 6: CREATE AI PROMPT
        # ----------------------------------------------------

        prompt = f"""
You are SNAP2MEAL, an expert AI chef.

Analyze the uploaded pantry photo.

### STEP 1: IDENTIFIED INGREDIENTS

- List ONLY items clearly visible in the photo.
- Also incorporate these manually added/corrected
  ingredients:

{extra_text}


### STEP 2: STAPLES & SHOPPING LIST

- List common missing household items such as
  oil, salt, water, rice, etc.
- Provide a shopping checklist for optional
  ingredients that improve the recipes.


### STEP 3: QUICK RECIPE OVERVIEW

Provide a Markdown table containing 3 distinct recipes.

| Recipe Name | Prep / Cook Time | Estimated Macros |
|---|---|---|
| Recipe 1 | 10m / 20m | ~300 kcal |
| Recipe 2 | 15m / 25m | ~350 kcal |
| Recipe 3 | 10m / 10m | ~250 kcal |


### STEP 4: DETAILED RECIPES

IMPORTANT RULES:

1. Cuisine Style:
   Adapt recipes to {cuisine_text} cuisine.

2. Do NOT invent main ingredients that are not
   visible in the image or explicitly added.

3. Use ONLY:
   - Identified pantry items
   - User-added ingredients
   - Common household staples

4. Scale all quantities for {servings} serving(s).

5. Follow these preferences:

Dietary Preference:
{pref_text}

Meal Type:
{types_text}


Format every recipe as:

#### 🍽️ Recipe Name

* Prep Time: X mins
* Cook Time: Y mins
* Difficulty: Easy/Medium

* Total Estimated Nutrition:
  ~X kcal | Yg Protein

* Ingredients Needed:

  - Item 1
  - Item 2

* Step-by-Step Instructions:

  1. Step one
  2. Step two
  3. Step three
"""


        # ----------------------------------------------------
        # STEP 7: SEND PROMPT + IMAGE TO OPENROUTER
        # ----------------------------------------------------

        response = client.chat.completions.create(

            model="openrouter/free",

            messages=[
                {
                    "role": "user",

                    "content": [

                        {
                            "type": "text",
                            "text": prompt
                        },

                        {
                            "type": "image_url",

                            "image_url": {
                                "url":
                                f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]
        )


        # ----------------------------------------------------
        # STEP 8: GET AI RESPONSE
        # ----------------------------------------------------

        recipe_text = response.choices[0].message.content


        # ----------------------------------------------------
        # STEP 9: CREATE DOWNLOADABLE FILE
        # ----------------------------------------------------

        temp_file = tempfile.NamedTemporaryFile(
            delete=False,
            mode="w",
            suffix=".txt"
        )

        temp_file.write(recipe_text)
        temp_file.close()


        # ----------------------------------------------------
        # STEP 10: RETURN RESULTS
        # ----------------------------------------------------

        return recipe_text, temp_file.name


    # --------------------------------------------------------
    # ERROR HANDLING
    # --------------------------------------------------------

    except Exception as e:

        return f"❌ Error: {str(e)}", None


print("✅ AI analysis function is ready")

✅ AI analysis function is ready


In [6]:
# ============================================================
# CELL 5: BUILD GRADIO USER INTERFACE
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css
) as demo:

    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    gr.Markdown("# 🥗 SNAP2MEAL")

    gr.Markdown(
        "Upload a snap of your pantry to generate "
        "instant recipes, customize ingredients, "
        "or ask Chef AI!"
    )


    # --------------------------------------------------------
    # CREATE TAB
    # --------------------------------------------------------

    with gr.Tabs():

        with gr.TabItem("📸 Recipe Generator"):

            # ------------------------------------------------
            # TWO-COLUMN LAYOUT
            # ------------------------------------------------

            with gr.Row():

                # ============================================
                # LEFT SIDE: INPUTS
                # ============================================

                with gr.Column(scale=1):

                    # Pantry image
                    image_input = gr.Image(
                        type="numpy",
                        label="📷 Upload Pantry Photo"
                    )


                    # Add or correct ingredients
                    extra_ingredients_input = gr.Textbox(
                        label="✏️ Add / Correct Ingredients (Optional)",

                        placeholder=(
                            "e.g., Add 2 eggs, 1 onion, "
                            "or replace ghee with butter..."
                        )
                    )


                    # Cuisine selection
                    cuisine_input = gr.Dropdown(
                        choices=[
                            "Any Cuisine",
                            "Indian",
                            "Italian",
                            "Mexican",
                            "Asian / Fusion",
                            "Mediterranean",
                            "American"
                        ],

                        value="Any Cuisine",

                        label="🌍 Preferred Cuisine Style"
                    )


                    # Dietary preference
                    dietary_input = gr.Dropdown(
                        choices=[
                            "None",
                            "Vegetarian",
                            "Vegan",
                            "Gluten-Free",
                            "High-Protein",
                            "Keto",
                            "Non-Vegetarian"
                        ],

                        value="Vegetarian",

                        label="🥗 Dietary Preference"
                    )


                    # Meal type
                    meal_type_input = gr.CheckboxGroup(
                        choices=[
                            "Breakfast",
                            "Main Course",
                            "Quick Snack / Side",
                            "Soup / Salad"
                        ],

                        value=["Main Course"],

                        label="🍽️ Preferred Meal Type"
                    )


                    # Number of servings
                    servings_input = gr.Slider(
                        minimum=1,
                        maximum=6,
                        value=2,
                        step=1,

                        label="👥 Number of Servings"
                    )


                    # Generate button
                    submit_btn = gr.Button(
                        "Generate Recipes ⚡",
                        variant="primary"
                    )


                # ============================================
                # RIGHT SIDE: OUTPUT
                # ============================================

                with gr.Column(scale=2):

                    # AI generated recipes
                    output_text = gr.Markdown(
                        label="Suggested Recipes",
                        elem_classes=["markdown-text"]
                    )


                    # Download recipe file
                    download_btn = gr.File(
                        label="📥 Download Recipes (.txt)"
                    )


    # --------------------------------------------------------
    # CONNECT BUTTON TO FUNCTION
    # --------------------------------------------------------

    submit_btn.click(

        fn=analyze_pantry,

        inputs=[
            image_input,
            extra_ingredients_input,
            cuisine_input,
            dietary_input,
            meal_type_input,
            servings_input
        ],

        outputs=[
            output_text,
            download_btn
        ]
    )


print("✅ Gradio interface created")

/tmp/ipykernel_1239/1465316730.py:5: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Gradio interface created


In [7]:
# ============================================================
# CELL 6: LAUNCH SNAP2MEAL
# ============================================================

demo.launch(
    inline=True,
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c9f5f1ca8a0b0545e0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
